# Task 5 – Bước 1: Tải dữ liệu FreshRetailNet-50K và kiểm tra cấu trúc
Notebook này dùng để:
1. Tải `train.parquet` và `eval.parquet` từ HuggingFace (Dingdong-Inc/FreshRetailNet-50K).
2. Đọc dữ liệu tiết kiệm RAM (các cột theo giờ đưa về mảng numpy 24 cột).
3. Kiểm tra cấu trúc: số dòng, kiểu dữ liệu, giá trị thiếu, khoảng thời gian, số thành phố/cửa hàng/sản phẩm/chuỗi.
4. **Xác định cách mã hóa `hours_stock_status`** (1 = hết hàng hay còn hàng?) và khung giờ của `stock_hour6_22_cnt`.
5. Kiểm tra eval **chỉ về cấu trúc và độ bao phủ** (không phân tích sâu, không dùng để chọn mô hình).
6. Lưu dữ liệu đã xử lý để các notebook EDA sau dùng lại.

> Chạy lần lượt từng ô từ trên xuống (Shift + Enter). Không cần GPU.

## 0. (Khuyến nghị) Kết nối Google Drive
Lưu dữ liệu vào Drive để lần sau không phải tải lại. Nếu không muốn, đặt `USE_DRIVE = False`.

In [1]:
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/FRN50K'
else:
    BASE_DIR = '/content/FRN50K'

RAW_DIR = os.path.join(BASE_DIR, 'raw')        # file parquet gốc
PROC_DIR = os.path.join(BASE_DIR, 'processed') # dữ liệu đã xử lý cho EDA
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
print('Thư mục dữ liệu:', BASE_DIR)

Mounted at /content/drive
Thư mục dữ liệu: /content/drive/MyDrive/FRN50K


## 1. Tải file parquet từ HuggingFace

In [2]:
from huggingface_hub import hf_hub_download

REPO_ID = 'Dingdong-Inc/FreshRetailNet-50K'
paths = {}
for split in ['train', 'eval']:
    paths[split] = hf_hub_download(
        repo_id=REPO_ID,
        repo_type='dataset',
        filename=f'data/{split}.parquet',
        local_dir=RAW_DIR,
    )
    size_mb = os.path.getsize(paths[split]) / 1024**2
    print(f'{split}: {paths[split]}  ({size_mb:.1f} MB)')

data/train.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/train.parquet: downloading bytes:           |  0.00B            

train: /content/drive/MyDrive/FRN50K/raw/data/train.parquet  (101.5 MB)


data/eval.parquet: reconstructing file:   0%|          |  0.00B / 8.44MB            

data/eval.parquet: downloading bytes:           |  0.00B            

eval: /content/drive/MyDrive/FRN50K/raw/data/eval.parquet  (8.0 MB)


## 2. Xem schema (chưa đọc dữ liệu vào RAM)

In [3]:
import pyarrow.parquet as pq

for split, p in paths.items():
    pf = pq.ParquetFile(p)
    print(f'===== {split}: {pf.metadata.num_rows:,} dòng, {pf.metadata.num_columns} cột =====')
    print(pf.schema_arrow)
    print()

===== train: 4,500,000 dòng, 19 cột =====
city_id: int64
store_id: int64
management_group_id: int64
first_category_id: int64
second_category_id: int64
third_category_id: int64
product_id: int64
dt: string
sale_amount: double
hours_sale: list<element: double>
  child 0, element: double
stock_hour6_22_cnt: int32
hours_stock_status: list<element: int64>
  child 0, element: int64
discount: double
holiday_flag: int32
activity_flag: int32
precpt: double
avg_temperature: double
avg_humidity: double
avg_wind_level: double
-- schema metadata --
huggingface: '{"info": {"features": {"city_id": {"dtype": "int64", "_type' + 1081

===== eval: 350,000 dòng, 19 cột =====
city_id: int64
store_id: int64
management_group_id: int64
first_category_id: int64
second_category_id: int64
third_category_id: int64
product_id: int64
dt: string
sale_amount: double
hours_sale: list<element: double>
  child 0, element: double
stock_hour6_22_cnt: int32
hours_stock_status: list<element: int64>
  child 0, element: int64

## 3. Đọc dữ liệu train
- Các cột một giá trị/ngày → `pandas.DataFrame` (`df`).
- `hours_sale` và `hours_stock_status` (dãy theo giờ) → mảng numpy `sale_h`, `stock_h` kích thước (số dòng × 24). Cách này nhẹ hơn nhiều so với để dạng list trong pandas.

In [4]:
import numpy as np
import pandas as pd
import pyarrow.compute as pc

HOURLY_COLS = ['hours_sale', 'hours_stock_status']

def load_split(path):
    schema = pq.read_schema(path)
    scalar_cols = [c for c in schema.names if c not in HOURLY_COLS]
    df = pq.read_table(path, columns=scalar_cols).to_pandas()

    hourly = {}
    for col in HOURLY_COLS:
        arr = pq.read_table(path, columns=[col]).column(col).combine_chunks()
        lens = pc.list_value_length(arr).to_numpy(zero_copy_only=False)
        print(f'{col}: độ dài dãy theo giờ ->', pd.Series(lens).value_counts().to_dict())
        vals = arr.flatten().to_numpy(zero_copy_only=False)
        L = int(lens[0])
        assert (lens == L).all(), 'Các dòng có độ dài khác nhau, cần xử lý riêng'
        hourly[col] = vals.reshape(-1, L)
    return df, hourly

df, hourly = load_split(paths['train'])
sale_h  = hourly['hours_sale'].astype('float32')
stock_h = hourly['hours_stock_status'].astype('int8')
del hourly

# Giảm bộ nhớ cho các cột id
for c in df.columns:
    if c.endswith('_id') or c.endswith('_flag'):
        df[c] = pd.to_numeric(df[c], downcast='integer')
df['dt'] = pd.to_datetime(df['dt'])

print(df.shape, sale_h.shape, stock_h.shape)
print(f'RAM DataFrame: {df.memory_usage(deep=True).sum()/1024**2:.0f} MB')
df.head()

hours_sale: độ dài dãy theo giờ -> {24: 4500000}
hours_stock_status: độ dài dãy theo giờ -> {24: 4500000}
(4500000, 17) (4500000, 24) (4500000, 24)
RAM DataFrame: 309 MB


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,stock_hour6_22_cnt,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,0,5,6,65,38,2024-03-28,0.1,0,1.0,0,0,1.6999,15.48,73.54,1.97
1,0,0,0,5,6,65,38,2024-03-29,0.1,1,1.0,0,0,3.0190,15.08,76.56,1.71
2,0,0,0,5,6,65,38,2024-03-30,0.0,0,1.0,1,0,2.0942,15.91,76.47,1.73
3,0,0,0,5,6,65,38,2024-03-31,0.1,11,1.0,1,0,1.5618,16.13,77.40,1.76
4,0,0,0,5,6,65,38,2024-04-01,0.2,8,1.0,0,0,3.5386,15.37,78.26,1.25


## 4. Kiểm tra kiểu dữ liệu, giá trị thiếu, miền giá trị

In [5]:
info = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'so_gia_tri_thieu': df.isna().sum(),
    'so_gia_tri_khac_nhau': df.nunique(),
    'min': df.min(numeric_only=False),
    'max': df.max(numeric_only=False),
})
print('NaN trong hours_sale:', int(np.isnan(sale_h).sum()))
print('Giá trị xuất hiện trong hours_stock_status:', np.unique(stock_h))
info

NaN trong hours_sale: 0
Giá trị xuất hiện trong hours_stock_status: [0 1]


,dtype,so_gia_tri_thieu,so_gia_tri_khac_nhau,min,max
city_id,int8,0,18,0,17
store_id,int16,0,898,0,897
management_group_id,int8,0,7,0,6
first_category_id,int8,0,32,0,31
second_category_id,int8,0,84,0,83
third_category_id,int16,0,233,0,232
product_id,int16,0,865,0,864
dt,datetime64[ns],0,90,2024-03-28 00:00:00,2024-06-25 00:00:00
sale_amount,float64,0,1363,0.0,44.9
stock_hour6_22_cnt,int32,0,17,0,16


## 5. Quy mô dữ liệu: thành phố, cửa hàng, sản phẩm, chuỗi, khoảng thời gian
Một **chuỗi** = một cặp (`store_id`, `product_id`).

In [6]:
n_days = df['dt'].nunique()
scale = pd.Series({
    'Số dòng (cửa hàng–sản phẩm–ngày)': len(df),
    'Số thành phố': df['city_id'].nunique(),
    'Số cửa hàng': df['store_id'].nunique(),
    'Số sản phẩm': df['product_id'].nunique(),
    'Số cặp cửa hàng–sản phẩm (chuỗi)': df.groupby(['store_id', 'product_id']).ngroups,
    'Số danh mục cấp 1': df['first_category_id'].nunique(),
    'Số danh mục cấp 2': df['second_category_id'].nunique(),
    'Số danh mục cấp 3': df['third_category_id'].nunique(),
    'Ngày đầu': df['dt'].min().date(),
    'Ngày cuối': df['dt'].max().date(),
    'Số ngày khác nhau': n_days,
}, name='train')
display(scale.to_frame())

# Mỗi chuỗi có đủ số ngày không? Có ngày bị thiếu giữa chừng không?
len_per_series = df.groupby(['store_id', 'product_id']).size()
print('Số ngày mỗi chuỗi:', len_per_series.value_counts().head(10).to_dict())
full_range = pd.date_range(df['dt'].min(), df['dt'].max())
print('Có ngày nào bị thiếu trong cả tập không?', len(full_range) != n_days)
print('Mỗi store_id thuộc đúng 1 city_id?', (df.groupby('store_id')['city_id'].nunique() == 1).all())

,train
Số dòng (cửa hàng–sản phẩm–ngày),4500000
Số thành phố,18
Số cửa hàng,898
Số sản phẩm,865
Số cặp cửa hàng–sản phẩm (chuỗi),50000
Số danh mục cấp 1,32
Số danh mục cấp 2,84
Số danh mục cấp 3,233
Ngày đầu,2024-03-28
Ngày cuối,2024-06-25


Số ngày mỗi chuỗi: {90: 50000}
Có ngày nào bị thiếu trong cả tập không? False
Mỗi store_id thuộc đúng 1 city_id? True


## 6. Xác định cách mã hóa `hours_stock_status` và khung giờ
README chỉ ghi `stock_hour6_22_cnt` = "số giờ hết hàng trong khoảng 6:00 đến 22:00", nhưng không nói rõ:
- giá trị 1 nghĩa là **hết hàng** hay **còn hàng** (bài báo quy ước s=1 là còn hàng),
- khung giờ là 6→21 (16 giờ) hay 6→22 (17 giờ).

Ô dưới thử mọi tổ hợp và xem tổ hợp nào khớp với `stock_hour6_22_cnt` gần 100%.

In [7]:
target = df['stock_hour6_22_cnt'].to_numpy()
rows = []
for lo, hi, label in [(6, 22, 'giờ 6..21 (16 giờ)'), (6, 23, 'giờ 6..22 (17 giờ)')]:
    for code_val in [1, 0]:
        cnt = (stock_h[:, lo:hi] == code_val).sum(axis=1)
        rows.append({'khung_gio': label,
                     'dem_gia_tri': code_val,
                     'ty_le_khop': (cnt == target).mean()})
check = pd.DataFrame(rows).sort_values('ty_le_khop', ascending=False)
display(check)

best = check.iloc[0]
print(f'=> Kết luận tạm: giá trị {best.dem_gia_tri} = HẾT HÀNG, khung giờ {best.khung_gio}, '
      f'khớp {best.ty_le_khop:.2%} số dòng.')
print('Nhớ ghi kết luận này vào notebook và vào danh sách lưu ý khi tái lập.')

,khung_gio,dem_gia_tri,ty_le_khop
0,giờ 6..21 (16 giờ),1,1.000000
2,giờ 6..22 (17 giờ),1,0.580815
1,giờ 6..21 (16 giờ),0,0.024848
3,giờ 6..22 (17 giờ),0,0.023147


=> Kết luận tạm: giá trị 1 = HẾT HÀNG, khung giờ giờ 6..21 (16 giờ), khớp 100.00% số dòng.
Nhớ ghi kết luận này vào notebook và vào danh sách lưu ý khi tái lập.


## 7. Kiểm tra `sale_amount` có bằng tổng `hours_sale` không

In [8]:
daily_from_hours = sale_h.sum(axis=1)
ok = np.isclose(daily_from_hours, df['sale_amount'].to_numpy(), rtol=1e-3, atol=1e-6)
print(f'Tỷ lệ dòng sale_amount ≈ tổng 24 giờ: {ok.mean():.2%}')
print('Sai lệch tuyệt đối lớn nhất:', float(np.abs(daily_from_hours - df['sale_amount']).max()))
print('Có doanh số âm không?', bool((sale_h < 0).any()))

Tỷ lệ dòng sale_amount ≈ tổng 24 giờ: 100.00%
Sai lệch tuyệt đối lớn nhất: 4.577636715907829e-06
Có doanh số âm không? False


## 8. Eval: chỉ kiểm tra cấu trúc và độ bao phủ
Không phân tích phân phối hay rút kết luận từ eval để tránh rò rỉ thông tin.

In [10]:
df_ev, hourly_ev = load_split(paths['eval'])
df_ev['dt'] = pd.to_datetime(df_ev['dt'])
del hourly_ev

key_tr = set(zip(df['store_id'], df['product_id']))
key_ev = set(zip(df_ev['store_id'], df_ev['product_id']))

coverage = pd.Series({
    'Số dòng eval': len(df_ev),
    'Cột giống train?': list(df_ev.columns) == list(df.columns),
    'Ngày đầu eval': df_ev['dt'].min().date(),
    'Ngày cuối eval': df_ev['dt'].max().date(),
    'Số ngày eval': df_ev['dt'].nunique(),
    'Số chuỗi eval': len(key_ev),
    'Tỷ lệ chuỗi eval có trong train': len(key_ev & key_tr) / len(key_ev),
    'Số ngày trùng giữa train và eval': len(set(df['dt']) & set(df_ev['dt'])),
    'Giá trị thiếu trong eval': int(df_ev.isna().sum().sum()),
}, name='eval')
coverage.to_frame()

hours_sale: độ dài dãy theo giờ -> {24: 350000}
hours_stock_status: độ dài dãy theo giờ -> {24: 350000}


,eval
Số dòng eval,350000
Cột giống train?,True
Ngày đầu eval,2024-06-26
Ngày cuối eval,2024-07-02
Số ngày eval,7
Số chuỗi eval,50000
Tỷ lệ chuỗi eval có trong train,1.0
Số ngày trùng giữa train và eval,0
Giá trị thiếu trong eval,0


## 9. Lưu dữ liệu train đã xử lý để dùng cho các notebook EDA tiếp theo

In [12]:
df.to_parquet(os.path.join(PROC_DIR, 'train_daily.parquet'), index=False)
np.save(os.path.join(PROC_DIR, 'train_hours_sale.npy'), sale_h)
np.save(os.path.join(PROC_DIR, 'train_hours_stock.npy'), stock_h)
scale.astype(str).to_frame().to_csv(os.path.join(PROC_DIR, 'bang_quy_mo_train.csv'))
coverage.astype(str).to_frame().to_csv(os.path.join(PROC_DIR, 'bang_bao_phu_eval.csv'))
check.to_csv(os.path.join(PROC_DIR, 'kiem_tra_ma_hoa_stock.csv'), index=False)
print('Đã lưu vào', PROC_DIR)
print(os.listdir(PROC_DIR))

Đã lưu vào /content/drive/MyDrive/FRN50K/processed
['train_daily.parquet', 'train_hours_sale.npy', 'train_hours_stock.npy', 'bang_quy_mo_train.csv', 'bang_bao_phu_eval.csv', 'kiem_tra_ma_hoa_stock.csv']


## Ghi chú kết quả

- **Quy mô (train):** 4.500.000 dòng (cửa hàng–sản phẩm–ngày); 18 thành phố, 898 cửa hàng, 865 sản phẩm, 50.000 chuỗi cửa hàng–sản phẩm; danh mục cấp 1/2/3: 32/84/233. Mỗi chuỗi có đủ 90 ngày, không thiếu ngày; mỗi cửa hàng thuộc đúng 1 thành phố. Không có giá trị thiếu.
- **Khoảng thời gian:** train 28/03/2024 – 25/06/2024 (90 ngày); eval 26/06/2024 – 02/07/2024 (7 ngày, ngay sau train).
- **Mã hóa `hours_stock_status`:** 1 = hết hàng, 0 = còn hàng (ngược với ký hiệu s=1 là còn hàng trong bài báo). `stock_hour6_22_cnt` đếm số giờ hết hàng trong giờ 6..21 (16 giờ, không gồm giờ 22), khớp 100%.
- **`sale_amount` và `hours_sale`:** `sale_amount` = tổng 24 giờ của `hours_sale` ở 100% số dòng (sai lệch tối đa khoảng 5e-6 do làm tròn); không có doanh số âm. Doanh số đã chuẩn hóa, không có đơn vị thật.
- **Eval:** 350.000 dòng = 50.000 chuỗi × 7 ngày; cột giống train; 100% chuỗi eval có trong train; không trùng ngày với train; không có giá trị thiếu.
- **Điểm bất thường cần kiểm tra thêm:** `discount` có giá trị 0.0 và giá trị > 1 (tối đa 1.088); số sản phẩm đếm được là 865, bài báo ghi 863.